In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# Load genome statistics with partition information
df_statistics = pd.read_csv("../../data/processed_data/dataset_information/genomes_info_with_partitions.csv")
df_statistics = df_statistics.rename(columns={"Unnamed: 0":"accession"})

In [ ]:
# Filter to test set only
df_testset = df_statistics[df_statistics['partition'] == 'test']

In [ ]:
# Define benchmark "reference" genomes
test_organisms = ["GCF_000011125.1", "GCF_000008665.1", "GCF_004799605.1", "GCF_000017165.1", "GCF_000007345.1", "GCF_000012545.1",
                  "GCF_000007365.1", "GCF_000009045.1", "GCF_025998455.1", "GCF_000195955.2", "GCF_020736045.1", "GCF_000012765.1",
                  "GCF_028609885.1", "GCF_000005845.2", "GCF_000006765.1", "GCF_000013425.1"]

# Get statistics for selected test organisms 

In [ ]:
# Collect data for TeX table to write
table_data = []
families = []
for accession in test_organisms:
    df_org = df_testset[df_testset['accession'] == accession]
    
    strain_name = df_org['strain'].values[0]
    species_name = df_org['species'].values[0]
    family_name = df_org['family'].values[0]
    gc_content = df_org['gc_content'].values[0]
    genome_size_kb = df_org['genome_length_kb'].values[0]
    genome_size_mbp = genome_size_kb / 1000  # Convert to Mbp
    cds_genes = df_org['protein_coding_genes'].values[0]
    coding_percentage = df_org['coding_percentage'].values[0]

    families.append(family_name)
    
    # Use strain if available, otherwise species
    if pd.isna(strain_name) or strain_name == 'nan':
        genome_name = f"{species_name} ({accession})"
    else:
        genome_name = f"{strain_name} ({accession})"
    
    table_data.append({
        'Genome': genome_name,
        'Size (Mbp)': f"{genome_size_mbp:.2f}",
        'Genes (CDS)': int(cds_genes),
        'Coding (%)': f"{coding_percentage:.2f}",
        'GC (%)': f"{gc_content:.1f}",
        'Translation Table': 'None'
    })

# Create DataFrame
df_table = pd.DataFrame(table_data)

# Generate LaTeX table
latex_table = df_table.to_latex(index=False, escape=False, column_format='llllll')

print(latex_table)

\begin{tabular}{llllll}
\toprule
Genome & Size (Mbp) & Genes (CDS) & Coding (%) & GC (%) & Translation Table \\
\midrule
Aeropyrum pernix K1 (GCF_000011125.1) & 1.67 & 1712 & 88.69 & 56.3 & None \\
Archaeoglobus fulgidus DSM 4304 (GCF_000008665.1) & 2.18 & 2515 & 92.61 & 48.6 & None \\
Halobacterium salinarum (GCF_004799605.1) & 2.43 & 2541 & 89.11 & 66.3 & None \\
Methanococcus vannielii SB (GCF_000017165.1) & 1.72 & 1725 & 86.67 & 31.3 & None \\
Methanosarcina acetivorans C2A (GCF_000007345.1) & 5.75 & 4884 & 75.08 & 42.7 & None \\
Methanosphaera stadtmanae DSM 3091 (GCF_000012545.1) & 1.77 & 1559 & 84.42 & 27.6 & None \\
Buchnera aphidicola str. Sg (Schizaphis graminum) (GCF_000007365.1) & 0.64 & 593 & 88.86 & 25.3 & None \\
Bacillus subtilis subsp. subtilis str. 168 (GCF_000009045.1) & 4.21 & 4240 & 87.34 & 43.5 & None \\
Helicobacter pylori (GCF_025998455.1) & 1.70 & 1584 & 91.54 & 38.5 & None \\
Mycobacterium tuberculosis H37Rv (GCF_000195955.2) & 4.41 & 3906 & 89.47 & 65.6 & Non

# Get statistics for each test family

In [ ]:
# Collect data for family table — two rows per family: median then range
rows = []

for family in sorted(set(families)):
    df_family = df_testset[df_testset['family'] == family]

    n_organisms = len(df_family)
    domain      = df_family['domain'].iloc[0]

    size_median = df_family['genome_length_kb'].median() / 1000
    size_min    = df_family['genome_length_kb'].min()    / 1000
    size_max    = df_family['genome_length_kb'].max()    / 1000

    genes_median = df_family['protein_coding_genes'].median()
    genes_min    = df_family['protein_coding_genes'].min()
    genes_max    = df_family['protein_coding_genes'].max()

    coding_median = df_family['coding_percentage'].median()
    coding_min    = df_family['coding_percentage'].min()
    coding_max    = df_family['coding_percentage'].max()

    gc_median = df_family['gc_content'].median()
    gc_min    = df_family['gc_content'].min()
    gc_max    = df_family['gc_content'].max()

    # Median row
    rows.append({
        'Family':             family,
        'Count':              n_organisms,
        'Size (Mbp)':         f"{size_median:.2f}",
        'Genes (CDS)':        f"{genes_median:.0f}",
        'Genome density (%)': f"{coding_median:.2f}",
        'GC (%)':             f"{gc_median:.1f}",
        'Translation Table':  'None',
        'Domain':             domain,
    })

    # Range row
    rows.append({
        'Family':             '',
        'Count':              '',
        'Size (Mbp)':         f"[{size_min:.2f}--{size_max:.2f}]",
        'Genes (CDS)':        f"[{genes_min:.0f}--{genes_max:.0f}]",
        'Genome density (%)': f"[{coding_min:.2f}--{coding_max:.2f}]",
        'GC (%)':             f"[{gc_min:.1f}--{gc_max:.1f}]",
        'Translation Table':  '',
        'Domain':             '',
    })

df_family_table = pd.DataFrame(rows)

latex_family_table = df_family_table.to_latex(index=False, escape=False, column_format='lrllllll')

print(latex_family_table)

\begin{tabular}{lrllllll}
\toprule
Family & Count & Size (Mbp) & Genes (CDS) & Genome density (%) & GC (%) & Translation Table & Domain \\
\midrule
Archaeoglobaceae & 3 & 2.18 & 2515 & 92.05 & 46.5 & None & Archaea \\
 &  & [1.90--2.20] & [2184--2573] & [91.99--92.61] & [44.1--48.6] &  &  \\
Bacillaceae & 22 & 4.22 & 4260 & 83.17 & 39.8 & None & Bacteria \\
 &  & [2.70--5.75] & [2702--5842] & [76.15--88.43] & [35.7--46.9] &  &  \\
Corynebacteriaceae & 33 & 2.52 & 2360 & 88.50 & 59.4 & None & Bacteria \\
 &  & [2.28--3.31] & [2019--3001] & [82.23--91.37] & [52.2--69.3] &  &  \\
Desulfurococcaceae & 6 & 1.35 & 1490 & 89.82 & 54.7 & None & Archaea \\
 &  & [1.30--1.67] & [1368--1712] & [88.69--92.49] & [44.8--56.7] &  &  \\
Enterobacteriaceae & 33 & 4.90 & 4555 & 88.10 & 54.6 & None & Bacteria \\
 &  & [4.36--6.30] & [3985--5842] & [85.31--89.80] & [49.9--58.0] &  &  \\
Erwiniaceae & 6 & 4.69 & 4262 & 86.30 & 54.7 & None & Bacteria \\
 &  & [0.64--4.80] & [593--4391] & [86.04--88.86] & [2